# Pandas Lab Notebook
### Hands-on practice to accompany *Pandas: The Complete Guide*

This notebook is meant to be **run cell by cell**. Each section has:
- a short recap
- a **📝 Try It** exercise cell (blank / partially blank — fill it in yourself)
- a **✅ Solution** cell right after (hidden in plain sight — try before you peek!)

Run the setup cell first — it builds a synthetic e-commerce dataset so the whole notebook works offline with no file downloads needed.

**How to use this in a training session:** the trainer runs through the recap + solution cells live; students clone the notebook and complete the 📝 cells independently, then compare against the ✅ cells.


## 0. Setup — Generate the Lab Dataset
Run this first. It creates two related tables — `orders` and `customers` — that we'll use throughout the notebook.

In [1]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)
np.random.seed(42)

# ---- customers table ----
n_customers = 40
cities = ["Pune", "Delhi", "Ahmedabad", "Bengaluru", "Mumbai", "Chennai"]
customers = pd.DataFrame({
    "customer_id": range(1, n_customers + 1),
    "name": [f"Customer_{i}" for i in range(1, n_customers + 1)],
    "city": np.random.choice(cities, n_customers),
    "signup_date": pd.to_datetime("2024-01-01") + pd.to_timedelta(np.random.randint(0, 500, n_customers), unit="D"),
    "age": np.random.randint(18, 65, n_customers)
})
# inject a few missing ages (real data is dirty!)
customers.loc[customers.sample(4, random_state=1).index, "age"] = np.nan

# ---- orders table ----
n_orders = 500
categories = ["Electronics", "Grocery", "Fashion", "Home", "Books"]
orders = pd.DataFrame({
    "order_id": range(1, n_orders + 1),
    "customer_id": np.random.randint(1, n_customers + 1, n_orders),
    "category": np.random.choice(categories, n_orders),
    "quantity": np.random.randint(1, 6, n_orders),
    "price": np.round(np.random.uniform(5, 500, n_orders), 2),
    "order_date": pd.to_datetime("2025-01-01") + pd.to_timedelta(np.random.randint(0, 365, n_orders), unit="D"),
    "status": np.random.choice(["Delivered", "Delivered", "Delivered", "Cancelled", "Returned"], n_orders)
})
# inject some duplicate rows and messy text (real data is dirty!)
orders = pd.concat([orders, orders.sample(5, random_state=2)], ignore_index=True)
orders["category"] = orders["category"].str.upper().where(orders.index % 17 == 0, orders["category"])

print("customers:", customers.shape, "| orders:", orders.shape)
customers.head()

AttributeError: module 'pandas._libs.algos' has no attribute 'take_1d_uint16_uint16'

In [ ]:
orders.head()

## 1. Exploration — Know Your Data
Always inspect before you analyze.

In [ ]:
# Recap
orders.info()
orders.describe()

### 📝 Try It
1. How many missing values does `customers` have, per column?
2. What are the unique values in `orders["status"]`, and their counts?
3. What percentage of `customers["age"]` is missing?


In [ ]:
# Your code here





### ✅ Solution

In [ ]:
print(customers.isna().sum())
print()
print(orders["status"].value_counts())
print()
print(f'{customers["age"].isna().mean()*100:.1f}% missing')

## 2. Indexing & Selection

In [ ]:
# Recap
orders.loc[orders["price"] > 400, ["order_id", "price", "category"]].head()
orders.iloc[0:3, 0:3]

### 📝 Try It
1. Select all orders from the "Electronics" category with `quantity >= 3`.
2. Select the `name` and `city` of customers older than 50 (watch out for NaN!).
3. Use `.query()` to find Cancelled orders priced above 100.


In [ ]:
# Your code here





### ✅ Solution

In [ ]:
q1 = orders[(orders["category"] == "Electronics") & (orders["quantity"] >= 3)]
q2 = customers.loc[customers["age"] > 50, ["name", "city"]]
q3 = orders.query("status == 'Cancelled' and price > 100")
q1.head(), q2.head(), q3.head()

## 3. Data Cleaning
Notice: we intentionally injected messy categories, duplicate rows, and missing ages in Section 0.

In [ ]:
# Recap — find the mess
print("Duplicate rows:", orders.duplicated().sum())
print("Category values:", orders["category"].unique())

### 📝 Try It
1. Remove duplicate rows from `orders`.
2. Standardize the `category` column so `"ELECTRONICS"` and `"Electronics"` are the same value.
3. Fill missing `age` values in `customers` with the **median** age.


In [ ]:
# Your code here





### ✅ Solution

In [ ]:
orders_clean = orders.drop_duplicates().copy()
orders_clean["category"] = orders_clean["category"].str.title()
customers_clean = customers.copy()
customers_clean["age"] = customers_clean["age"].fillna(customers_clean["age"].median())

print("Rows after dedup:", orders_clean.shape[0])
print("Categories now:", orders_clean["category"].unique())
print("Missing ages left:", customers_clean["age"].isna().sum())

## 4. Transformation

In [ ]:
# Recap
orders_clean["revenue"] = orders_clean["price"] * orders_clean["quantity"]
orders_clean["price_tier"] = np.where(orders_clean["price"] > 200, "Premium", "Standard")
orders_clean.head()

### 📝 Try It
1. Create a column `age_group` on `customers_clean`: `"Young"` (<30), `"Mid"` (30-50), `"Senior"` (>50).
2. Create a column `is_high_value` on `orders_clean` flagging orders where `revenue > 500`.
3. Bucket `orders_clean["price"]` into 4 equal-frequency bins using `pd.qcut`.


In [ ]:
# Your code here





### ✅ Solution

In [ ]:
customers_clean["age_group"] = pd.cut(customers_clean["age"], bins=[0, 30, 50, 100],
                                       labels=["Young", "Mid", "Senior"])
orders_clean["is_high_value"] = orders_clean["revenue"] > 500
orders_clean["price_quartile"] = pd.qcut(orders_clean["price"], q=4, labels=["Q1", "Q2", "Q3", "Q4"])
customers_clean[["name", "age", "age_group"]].head(), orders_clean[["price", "price_quartile"]].head()

## 5. GroupBy — Split, Apply, Combine

In [ ]:
# Recap
orders_clean.groupby("category")["revenue"].agg(["sum", "mean", "count"])

### 📝 Try It
1. Find total revenue per `status` (Delivered / Cancelled / Returned).
2. Find the top revenue-generating category **per city** — you'll need to merge `orders_clean` with `customers_clean` first.
3. Using `.transform()`, add a column showing each order's revenue as a % of its category's total revenue.


In [ ]:
# Your code here





### ✅ Solution

In [ ]:
q1 = orders_clean.groupby("status")["revenue"].sum()

merged = orders_clean.merge(customers_clean, on="customer_id", how="left")
q2 = (merged.groupby(["city", "category"])["revenue"].sum()
             .reset_index()
             .sort_values("revenue", ascending=False)
             .drop_duplicates("city"))

orders_clean["pct_of_category"] = (
    orders_clean["revenue"] / orders_clean.groupby("category")["revenue"].transform("sum") * 100
)

q1, q2.head(), orders_clean[["category", "revenue", "pct_of_category"]].head()

## 6. Merging Tables

In [ ]:
# Recap
merged = orders_clean.merge(customers_clean, on="customer_id", how="left", suffixes=("_order", "_cust"))
merged.head()

### 📝 Try It
1. Find customers who have **never placed an order** (hint: `how="right"` + `indicator=True`, or an anti-join pattern).
2. Confirm the merge didn't accidentally create duplicate rows by comparing row counts before/after.


In [ ]:
# Your code here





### ✅ Solution

In [ ]:
check = customers_clean.merge(orders_clean, on="customer_id", how="left", indicator=True)
never_ordered = check[check["_merge"] == "left_only"][["customer_id", "name"]].drop_duplicates()

print("Orders before merge:", orders_clean.shape[0], "| after merge:", merged.shape[0])
never_ordered

## 7. Reshaping — Pivot Tables

In [ ]:
# Recap
pd.pivot_table(merged, index="city", columns="category", values="revenue",
               aggfunc="sum", fill_value=0)

### 📝 Try It
1. Build a pivot table: rows = `status`, columns = `category`, values = **average** `price`.
2. Add row and column totals with `margins=True`.


In [ ]:
# Your code here





### ✅ Solution

In [ ]:
pd.pivot_table(merged, index="status", columns="category", values="price",
               aggfunc="mean", fill_value=0, margins=True)

## 8. Time Series

In [ ]:
# Recap
ts = orders_clean.set_index("order_date").sort_index()
monthly_revenue = ts.resample("ME")["revenue"].sum()
monthly_revenue.head()

### 📝 Try It
1. Compute a 7-day rolling average of daily revenue.
2. Find which month had the highest total revenue.
3. Add a `day_of_week` column derived from `order_date`.


In [ ]:
# Your code here





### ✅ Solution

In [ ]:
daily_revenue = ts.resample("D")["revenue"].sum()
rolling_7d = daily_revenue.rolling(7).mean()

best_month = monthly_revenue.idxmax()

orders_clean["day_of_week"] = orders_clean["order_date"].dt.day_name()

rolling_7d.tail(), best_month, orders_clean[["order_date", "day_of_week"]].head()

## 9. String Operations

In [ ]:
# Recap
customers_clean["city"].str.upper().head()

### 📝 Try It
1. Create a `name_initial` column with just the first letter of each customer's name.
2. Filter customers whose `city` contains the letter "a" (case-insensitive).


In [ ]:
# Your code here





### ✅ Solution

In [ ]:
customers_clean["name_initial"] = customers_clean["name"].str[0]
has_a = customers_clean[customers_clean["city"].str.contains("a", case=False)]
customers_clean[["name", "name_initial"]].head(), has_a[["name", "city"]].head()

## 10. Capstone Exercise — Mini RFM Analysis

Put everything together: build a **Recency / Frequency / Monetary** segmentation of customers, exactly as a real e-commerce analytics team would.

### 📝 Try It
1. For each `customer_id`, compute:
   - `recency` = days since their most recent order (relative to the day after the last order in the dataset)
   - `frequency` = number of distinct orders
   - `monetary` = total revenue
2. Split each metric into quartiles (`pd.qcut`) scored 1–4 (higher = better).
3. Combine the three scores into an `rfm_segment` string like `"432"`.
4. Find the top 5 customers by combined RFM score.


In [ ]:
# Your code here — take your time, this combines everything above






### ✅ Solution

In [ ]:
snapshot_date = orders_clean["order_date"].max() + pd.Timedelta(days=1)

rfm = orders_clean.groupby("customer_id").agg(
    recency=("order_date", lambda x: (snapshot_date - x.max()).days),
    frequency=("order_id", "nunique"),
    monetary=("revenue", "sum")
).reset_index()

rfm["r_score"] = pd.qcut(rfm["recency"], 4, labels=[4, 3, 2, 1]).astype(int)   # lower recency = better = higher score
rfm["f_score"] = pd.qcut(rfm["frequency"].rank(method="first"), 4, labels=[1, 2, 3, 4]).astype(int)
rfm["m_score"] = pd.qcut(rfm["monetary"], 4, labels=[1, 2, 3, 4]).astype(int)

rfm["rfm_segment"] = rfm["r_score"].astype(str) + rfm["f_score"].astype(str) + rfm["m_score"].astype(str)
rfm["total_score"] = rfm[["r_score", "f_score", "m_score"]].sum(axis=1)

rfm.merge(customers_clean[["customer_id", "name"]], on="customer_id") \
   .sort_values("total_score", ascending=False) \
   .head(5)[["customer_id", "name", "recency", "frequency", "monetary", "rfm_segment", "total_score"]]

---
## Wrap-Up

You've now practiced every core Pandas skill on a realistic, messy dataset:
exploration → indexing → cleaning → transformation → groupby → merging → reshaping → time series → strings → a full capstone analysis.

**Next steps for students:**
- Re-run the capstone with a different snapshot date and see how segments shift.
- Try swapping `pd.cut` bins in Section 4 for different thresholds.
- Load your own CSV with `pd.read_csv()` and repeat this whole workflow end-to-end.
